# Scaricare e Formattare Dati da Overpass API

Questo notebook scarica dati dai punti di interesse (POI) utilizzando l'Overpass API di OpenStreetMap e li formatta in un DataFrame pandas.

In [ ]:
import requests
import json
import pandas as pd

In [ ]:
# Leggere la query da pois_query.txt
with open('pois_query.txt', 'r') as file:
    pois_query = file.read()

print("Query caricata:")
print(pois_query[:200] + "..." if len(pois_query) > 200 else pois_query)

In [ ]:
# Inviare la richiesta all'Overpass API o caricare da file esistente
import os

if os.path.exists('pois.json'):
    print("File pois.json già presente, caricamento dati locali.")
    with open('pois.json', 'r') as f:
        data = json.load(f)
else:
    overpass_url = "http://overpass-api.de/api/interpreter"
    response = requests.post(overpass_url, data={'data': pois_query})

    if response.status_code == 200:
        data = response.json()
        print("Dati scaricati con successo.")
        # Salva per future esecuzioni
        with open('pois.json', 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        print("Dati salvati in pois.json")
    else:
        print(f"Errore nella richiesta: {response.status_code}")
        data = None

In [ ]:
# Importa le configurazioni di amenity
import sys
sys.path.insert(0, '../..')
from amenities_config import CATEGORIES, CATEGORY_AMENITIES

category_amenities = CATEGORY_AMENITIES

In [ ]:
# Formattare i dati in un DataFrame con logica per ways e relations
if data:
    elements = data.get('elements', [])
    
    # Separare elementi per tipo
    nodes = {elem['id']: elem for elem in elements if elem['type'] == 'node'}
    ways = {elem['id']: elem for elem in elements if elem['type'] == 'way'}
    relations = {elem['id']: elem for elem in elements if elem['type'] == 'relation'}
    
    def clean_amenity_value(value):
        if not isinstance(value, str):
            return value
        return value.split(';', 1)[0].strip()
    
    def round_coord(value):
        if value is None:
            return None
        return round(value, 7)
    
    # Funzione per ottenere categoria e amenity
    def get_category_amenity(tags):
        tags = tags or {}
        for category, amenities in category_amenities.items():
            for amenity_key, amenity_value in amenities:
                if (amenity_value == '*' and amenity_key in tags) or tags.get(amenity_key) == amenity_value:
                    amenity_value_real = tags.get(amenity_key) if amenity_value == '*' else amenity_value
                    amenity_value_real = clean_amenity_value(amenity_value_real)
                    return category, amenity_value_real
        return None, None
    
    def is_yes_amenity(value):
        return value == 'yes'
    
    # Lista finale dei POI
    pois = []
    used_node_ids = set()
    
    # Prima, elabora nodi validi
    for nid, node in nodes.items():
        if node.get('lat') is not None and node.get('lon') is not None:
            tags = node.get('tags')
            if not tags:
                continue
            cat, am = get_category_amenity(tags)
            if is_yes_amenity(am):
                continue
            pois.append({
                'id': nid,
                'type': 'node',
                'lat': round_coord(node['lat']),
                'lon': round_coord(node['lon']),
                'category': cat,
                'amenity': am
            })
            used_node_ids.add(nid)
    
    # Poi, elabora ways
    for wid, way in ways.items():
        valid_node_ids = [nid for nid in way['nodes'] if nid in nodes and nid not in used_node_ids and nodes[nid].get('lat') is not None]
        if not valid_node_ids:
            continue
        node_infos = []
        processed_node_ids = []
        for nid in valid_node_ids:
            n = nodes[nid]
            cat, am = get_category_amenity(n.get('tags', {}))
            if is_yes_amenity(am):
                continue
            node_infos.append((cat, am, n))
            processed_node_ids.append(nid)
        if not processed_node_ids:
            continue
        final_cat = None
        final_am = None
        if node_infos:
            sample_cat, sample_am = node_infos[0][0], node_infos[0][1]
            consistent = all(info[0] == sample_cat and info[1] == sample_am for info in node_infos)
            if consistent and sample_cat is not None and sample_am is not None:
                final_cat, final_am = sample_cat, sample_am
        if final_cat is None or final_am is None:
            fallback_cat, fallback_am = get_category_amenity(way.get('tags', {}))
            if is_yes_amenity(fallback_am):
                continue
            final_cat, final_am = fallback_cat, fallback_am
        lats = [nodes[nid]['lat'] for nid in processed_node_ids]
        lons = [nodes[nid]['lon'] for nid in processed_node_ids]
        if not lats or not lons:
            continue
        lat_cent = round_coord(sum(lats) / len(lats))
        lon_cent = round_coord(sum(lons) / len(lons))
        pois.append({
            'id': wid,
            'type': 'way',
            'lat': lat_cent,
            'lon': lon_cent,
            'category': final_cat,
            'amenity': final_am
        })
        used_node_ids.update(processed_node_ids)
    
    # Infine, elabora relations
    for rid, rel in relations.items():
        member_pois = []
        for member in rel.get('members', []):
            if member['type'] == 'node':
                nid = member['ref']
                if nid in nodes and nid not in used_node_ids and nodes[nid].get('lat') is not None:
                    n = nodes[nid]
                    cat, am = get_category_amenity(n.get('tags', {}))
                    if is_yes_amenity(am):
                        continue
                    member_pois.append({
                        'id': nid,
                        'type': 'node',
                        'lat': round_coord(n['lat']),
                        'lon': round_coord(n['lon']),
                        'category': cat,
                        'amenity': am
                    })
                    used_node_ids.add(nid)
            elif member['type'] == 'way':
                wid = member['ref']
                way = ways.get(wid)
                if way:
                    valid_node_ids = [nid for nid in way['nodes'] if nid in nodes and nid not in used_node_ids and nodes[nid].get('lat') is not None]
                    if valid_node_ids:
                        node_infos = []
                        processed_node_ids = []
                        for nid in valid_node_ids:
                            n = nodes[nid]
                            cat, am = get_category_amenity(n.get('tags', {}))
                            if is_yes_amenity(am):
                                continue
                            node_infos.append((cat, am, n))
                            processed_node_ids.append(nid)
                        if processed_node_ids:
                            final_cat = None
                            final_am = None
                            if node_infos:
                                sample_cat, sample_am = node_infos[0][0], node_infos[0][1]
                                consistent = all(info[0] == sample_cat and info[1] == sample_am for info in node_infos)
                                if consistent and sample_cat is not None and sample_am is not None:
                                    final_cat, final_am = sample_cat, sample_am
                            if final_cat is None or final_am is None:
                                fallback_cat, fallback_am = get_category_amenity(way.get('tags', {}))
                                if is_yes_amenity(fallback_am):
                                    continue
                                final_cat, final_am = fallback_cat, fallback_am
                            lats = [nodes[nid]['lat'] for nid in processed_node_ids]
                            lons = [nodes[nid]['lon'] for nid in processed_node_ids]
                            if not lats or not lons:
                                continue
                            lat_cent = round_coord(sum(lats) / len(lats))
                            lon_cent = round_coord(sum(lons) / len(lons))
                            member_pois.append({
                                'id': wid,
                                'type': 'way',
                                'lat': lat_cent,
                                'lon': lon_cent,
                                'category': final_cat,
                                'amenity': final_am
                            })
                            used_node_ids.update(processed_node_ids)
        if member_pois:
            node_infos = [(p['category'], p['amenity']) for p in member_pois]
            final_cat = None
            final_am = None
            if node_infos:
                sample_cat, sample_am = node_infos[0]
                consistent = all(info[0] == sample_cat and info[1] == sample_am for info in node_infos)
                if consistent and sample_cat is not None and sample_am is not None:
                    final_cat, final_am = sample_cat, sample_am
            if final_cat is None or final_am is None:
                fallback_cat, fallback_am = get_category_amenity(rel.get('tags', {}))
                if is_yes_amenity(fallback_am):
                    continue
                final_cat, final_am = fallback_cat, fallback_am
            lats = [p['lat'] for p in member_pois]
            lons = [p['lon'] for p in member_pois]
            if not lats or not lons:
                continue
            lat_cent = round_coord(sum(lats) / len(lats))
            lon_cent = round_coord(sum(lons) / len(lons))
            pois.append({
                'id': rid,
                'type': 'relation',
                'lat': lat_cent,
                'lon': lon_cent,
                'category': final_cat,
                'amenity': final_am
            })
    
    df = pd.DataFrame(pois)
    print("DataFrame creato con", len(df), "elementi.")
    display(df.head())
else:
    print("Nessun dato da formattare.")

In [ ]:
# Pulire i dati: rimuovere elementi senza lat/lon
if 'df' in locals() and not df.empty:
    # Rimuovi righe con lat o lon nulli
    df = df.dropna(subset=['lat', 'lon'])
    print("Dopo pulizia:", len(df), "elementi rimanenti.")
    display(df.head())
else:
    print("Nessun DataFrame da pulire.")

In [ ]:
def prepare_pois_with_amenity(df):
    """
    Preprocessa i POI aggiungendo categorie e amenity basate sui tags.
    Ritorna un dizionario {categoria: {amenity: [pois]}}
    """
    pois_by_category = {cat: {} for cat in category_amenities.keys()}
    
    for _, row in df.iterrows():
        cat = row['category']
        am = row['amenity']
        if cat and am:
            if am not in pois_by_category[cat]:
                pois_by_category[cat][am] = []
            poi = {
                'id': row['id'],
                'lat': row['lat'],
                'lon': row['lon']
            }
            pois_by_category[cat][am].append(poi)
    
    return pois_by_category

In [ ]:
# Applicare la categorizzazione
if 'df' in locals() and not df.empty:
    pois_by_category = prepare_pois_with_amenity(df)
    print("POI per categoria e amenity:")
    for cat, amenities_dict in pois_by_category.items():
        print(f"{cat}:")
        for amenity, pois in amenities_dict.items():
            print(f"  {amenity}: {len(pois)} POI")
        print(f"  Totale {cat}: {sum(len(pois) for pois in amenities_dict.values())} POI")
else:
    print("Nessun DataFrame da categorizzare.")

In [ ]:
# Salvare i dati formattati
if 'pois_by_category' in locals():
    with open('pois_by_category.json', 'w', encoding='utf-8') as f:
        json.dump(pois_by_category, f, indent=2, ensure_ascii=False)
    print("Dati categorizzati salvati in pois_by_category.json")

In [ ]:
# Definire una classe per gli spostamenti
class Spostamento:
    def __init__(self, from_category, from_amenity, to_category, to_amenity):
        self.from_category = from_category
        self.from_amenity = from_amenity
        self.to_category = to_category
        self.to_amenity = to_amenity

# Definire la lista di spostamenti da una categoria all'altra
spostamenti = [
    Spostamento(
        from_category='commerciale',
        from_amenity='swimming_pool',
        to_category='sport',
        to_amenity='swimming_pool'
    )
    # Aggiungi altri spostamenti se necessario
]

In [ ]:
def applica_spostamenti(pois_by_category, spostamenti):
    """
    Applica gli spostamenti definiti alla struttura pois_by_category.
    """
    for spostamento in spostamenti:
        from_cat = spostamento.from_category
        from_amenity = spostamento.from_amenity
        to_cat = spostamento.to_category
        to_amenity = spostamento.to_amenity
        
        if from_cat in pois_by_category and from_amenity in pois_by_category[from_cat]:
            pois = pois_by_category[from_cat].pop(from_amenity)
            
            if to_amenity not in pois_by_category[to_cat]:
                pois_by_category[to_cat][to_amenity] = []
            pois_by_category[to_cat][to_amenity].extend(pois)
            print(f"Spostati {len(pois)} POI da {from_cat} ({from_amenity}) a {to_cat} ({to_amenity})")
    
    return pois_by_category

In [ ]:
# Applicare gli spostamenti
if 'pois_by_category' in locals() and 'spostamenti' in locals():
    pois_by_category = applica_spostamenti(pois_by_category, spostamenti)
    
    # Salvare i dati aggiornati
    with open('pois_by_category.json', 'w', encoding='utf-8') as f:
        json.dump(pois_by_category, f, indent=2, ensure_ascii=False)
    print("Dati aggiornati salvati in pois_by_category.json")
else:
    print("Nessun dato o spostamenti da applicare.")

In [ ]:
# Estrarre distinct amenity per categoria dai dati aggiornati
if 'pois_by_category' in locals():
    distinct_amenities_per_category = {cat: set() for cat in category_amenities.keys()}
    
    # Iterare sui POI categorizzati
    for cat, amenities_dict in pois_by_category.items():
        for amenity_value in amenities_dict:
            distinct_amenities_per_category[cat].add(amenity_value)
    
    # Creare un DataFrame per salvare in CSV
    rows = []
    for cat, amenities in distinct_amenities_per_category.items():
        for amenity_value in amenities:
            rows.append({'categoria': cat, 'amenity_value': amenity_value})
    df_amenities = pd.DataFrame(rows)
    df_amenities.to_csv('distinct_amenities_per_category.csv', index=False)
    print("Distinct amenities per categoria salvate in distinct_amenities_per_category.csv")
else:
    print("Nessun dato categorizzato per estrarre amenities.")

In [ ]:
# Stampare i POI del `df` che non sono presenti in `pois_by_category`
import os
import json

if 'df' not in locals() or df.empty:
    print("Nessun DataFrame `df` disponibile. Esegui le celle precedenti.")
else:
    # Carica pois_by_category da variabile o file
    if 'pois_by_category' not in locals():
        if os.path.exists('pois_by_category.json'):
            with open('pois_by_category.json', 'r', encoding='utf-8') as f:
                pois_by_category = json.load(f)
        else:
            print("Nessun `pois_by_category` disponibile (variabile o file).")
    if 'pois_by_category' in locals():
        ids_df = set(int(x) for x in df['id'].tolist())
        ids_pbc = set()
        for cat_dict in pois_by_category.values():
            for am_list in cat_dict.values():
                for poi in am_list:
                    try:
                        ids_pbc.add(int(poi['id']))
                    except Exception:
                        pass
        missing = ids_df - ids_pbc
        print(f"POI in df: {len(ids_df)}; in pois_by_category: {len(ids_pbc)}; missing: {len(missing)}")
        if missing:
            missing_list = sorted(missing)
            display(df[df['id'].isin(missing_list)])
            # Salva gli id mancanti su file TXT
            with open('missing_poi_ids.txt', 'w', encoding='utf-8') as f:
                for mid in missing_list:
                    f.write(f"{mid}\n")
            print(f"Saved {len(missing_list)} missing ids to missing_poi_ids.json and missing_poi_ids.txt")
            # Mostra informazioni originali dagli elementi di pois.json se presente
            if os.path.exists('pois.json'):
                with open('pois.json', 'r', encoding='utf-8') as f:
                    original = json.load(f)
                elements = original.get('elements', [])
                elem_map = {e['id']: e for e in elements}
                for mid in missing_list[:50]:
                    e = elem_map.get(mid)
                    if e:
                        print('-----')
                        print(f"id: {mid}, type: {e.get('type')}, tags keys: {list(e.get('tags', {}).keys())}")
                    else:
                        print(f"id: {mid} non trovato in pois.json")